In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import yfinance as yf
import cvxpy as cvx
import riskfolio as rf

import statsmodels.api as sm

import datetime as dt
from arch import arch_model

from datetime import date, timedelta

from statsmodels.graphics.tsaplots import plot_pacf, plot_acf

In [3]:
# --- 1. Download SPY data for the past ~6 months (to ensure enough data for training and testing) ---
TICKER = 'QQQ'
end_date = date.today()
# Get data for a longer period to train the model, say 6 months
start_date_train = end_date - timedelta(days=180) 
# Define the 3-month window for comparison
start_date_comparison = end_date - timedelta(days=90) 

print(f"Downloading {TICKER} data from {start_date_train} to {end_date}...")
data = yf.download(TICKER, start=start_date_train, end=end_date, auto_adjust=True)
print("Data downloaded successfully.")

[*********************100%***********************]  1 of 1 completed

1 Failed download:
['QQQ']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


Data downloaded successfully.


In [ ]:
# Calculate log returns
# GARCH models are typically applied to returns, not prices
returns = 100 * data['Adj Close'].pct_change().dropna()

# Split data into training and comparison sets
train_returns = returns[:start_date_comparison]
comparison_returns = returns[start_date_comparison:]

print(f"Training data points: {len(train_returns)}")
print(f"Comparison data points (past 3 months): {len(comparison_returns)}")

# --- 2. Fit GARCH model ---
# A basic GARCH(1,1) model is a common starting point
# Using a 'Normal' distribution for simplicity
model = arch_model(train_returns, vol='Garch', p=1, o=0, q=1, dist='Normal')
print("\nFitting GARCH model to training data...")
results = model.fit(disp='off') # disp='off' suppresses iteration details
print(results.summary())

# --- 3. Compare forecasted volatilities to historical ones on the past 3 month window ---

# Perform rolling 1-step ahead forecasts for the comparison period
forecasted_variances = []
# Start from the date right after the training set ends
current_date = comparison_returns.index[0] 

# Re-fit the model in a rolling window fashion (more realistic)
# This can be computationally intensive for large datasets/windows
for i in range(len(comparison_returns)):
    # Data up to the current date minus one day for 1-step ahead forecast
    train_subset = returns[:current_date]
    sub_model = arch_model(train_subset, vol='Garch', p=1, o=0, q=1, dist='Normal')
    sub_results = sub_model.fit(disp='off', last_obs=current_date)
    
    # Forecast 1 step ahead (horizon=1)
    forecast = sub_results.forecast(horizon=1, start=current_date)
    # Extract the forecasted variance for the next day
    next_day_variance = forecast.variance.iloc[-1, 0] # Get the value for 'h.1'
    forecasted_variances.append(next_day_variance)

    # Move to the next day
    if i + 1 < len(comparison_returns):
        current_date = comparison_returns.index[i + 1]

# Convert list to a pandas Series
forecasted_variances = pd.Series(forecasted_variances, index=comparison_returns.index)

# The 'historical' (realized) volatility is the squared returns in this context
historical_variances = comparison_returns**2 

# Annualize the volatilities for plotting (often useful for comparison)
# Assuming 252 trading days in a year
forecasted_volatility_annual = np.sqrt(forecasted_variances) * np.sqrt(252)
historical_volatility_annual = np.sqrt(historical_variances) * np.sqrt(252)

# --- 4. Visualization ---
plt.figure(figsize=(12, 6))
plt.plot(historical_volatility_annual.index, historical_volatility_annual, label='Historical Volatility (Annualized)', color='gray', alpha=0.6)
plt.plot(forecasted_volatility_annual.index, forecasted_volatility_annual, label='GARCH Forecasted Volatility (Annualized)', color='blue', linestyle='--')
plt.title(f'GARCH Forecasted vs Historical Volatility for {TICKER} (Past 3 Months)')
plt.xlabel('Date')
plt.ylabel('Annualized Volatility (%)')
plt.legend()
plt.grid(True)
plt.show()

# Print a brief comparison
mse = np.mean((forecasted_variances - historical_variances)**2)
print(f"\nMean Squared Error (MSE) between forecasted and historical variances: {mse:.4f}")

In [2]:
# 1. Download historical SPY data
# Using yfinance to get data
ticker = 'SPY'
end_date = pd.Timestamp.today()
start_date = end_date - pd.DateOffset(years=5) # Get last 5 years of data
spy_data = yf.download(ticker, start=start_date, end=end_date, auto_adjust=True)

[*********************100%***********************]  1 of 1 completed


In [3]:
spy_data.head()

Price,Close,High,Low,Open,Volume
Ticker,SPY,SPY,SPY,SPY,SPY
Date,,,,,
2020-12-18,345.639130,347.483513,343.616859,347.314998,136542300
2020-12-21,344.403259,354.327352,338.945024,341.697556,96386700
2020-12-22,343.822754,344.843246,342.689918,344.730903,47949000
2020-12-23,344.131836,346.051106,343.804148,344.796555,46201400
2020-12-24,345.470581,345.498667,344.019429,344.609233,26457900


In [4]:
spy_data.columns = [c[0] for c in spy_data.columns]

In [5]:
# Ensure data is sorted by date
spy_data.sort_index(inplace=True)

# 2. Calculate log returns (expressed in percentage to help the optimizer converge)
# Drop any missing values created by the shift operation
returns = 100 * spy_data['Close'].pct_change().dropna()

# Plot the returns to observe volatility clustering
# returns.plot(title="SPY Daily Returns (%)")
# plt.show()

# 3. Specify and fit the GARCH(1,1) model
# We use 'GARCH' for volatility model, 'constant' for mean model, 
# and 'Normal' for distribution, with p=1 and q=1
model = arch_model(returns, vol='Garch', p=1, o=0, q=1, mean='constant', dist='normal')
results = model.fit(disp='off') # 'disp="off"' prevents fit summary from printing during fitting

# Print the model summary to see parameters like alpha[1] and beta[1]
print(results.summary())

# 4. Make a 1-step ahead forecast
# The forecast is made using data up to and including the last date in the returns series
forecasts = results.forecast(horizon=1, start=returns.index[-1], method='analytic')

# The output of the forecast is a DataFrame of variances
# Get the variance for the next period (h.1)
# Square root the variance to get the conditional volatility
next_day_variance = forecasts.variance.iloc[-1, 0]
next_day_volatility = np.sqrt(next_day_variance)

print(f"\nLast observation date: {returns.index[-1].date()}")
print(f"Forecasted 1-day ahead conditional volatility for the next trading day: {next_day_volatility:.4f}%")

# To get annualized volatility, multiply by sqrt(252)
annualized_volatility = next_day_volatility * np.sqrt(252)
print(f"Forecasted 1-day ahead annualized volatility: {annualized_volatility:.4f}%")

                     Constant Mean - GARCH Model Results                      
Dep. Variable:                  Close   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -1713.42
Distribution:                  Normal   AIC:                           3434.85
Method:            Maximum Likelihood   BIC:                           3455.39
                                        No. Observations:                 1254
Date:                Thu, Dec 18 2025   Df Residuals:                     1253
Time:                        09:17:27   Df Model:                            1
                                Mean Model                                
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
mu             0.0876  2.350e-02      3.728  1.933e-04 [4.154e-0